In [ ]:
import requests
import json
import os
import time
from openai import OpenAI
from google.colab import userdata
import pandas as pd
import pprint

In [ ]:
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [ ]:
!pip install pinecone

In [ ]:
from pinecone import Pinecone

In [ ]:
pc = Pinecone(api_key='pcsk_6akU8Z_2BXXXDSBKbvFCn4sciNM2FeJC6PwAt6wFwQeQjoJKDSjysRbtyBAdUfRv6z87e6')
index = pc.Index('cus635')

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
model = SentenceTransformer("thenlper/gte-large")

In [ ]:
response = index.query(
    vector=[0.0] * model.get_sentence_embedding_dimension(),
    top_k=100,
    include_metadata=True,
    namespace="TEAM_3",
)

for match in response['matches']:
    print(f"ID: {match['id']}")
    print(f"Score: {match['score']}")
    print(f"Metadata: {match['metadata']}")
    print("="*40)

In [ ]:
retrieved_texts = []

for match in response['matches']:
    metadata = match.get('metadata', {})
    category = metadata.get('category', 'Unknown')
    text = metadata.get('text', '')

    raw_json_str = metadata.get('raw', '{}')
    try:
        article = json.loads(raw_json_str)
    except json.JSONDecodeError:
        article = {}

    title = article.get("webTitle", text)
    url = article.get("webUrl", "")
    pub_date = article.get("webPublicationDate", "")
    section = article.get("sectionName", "")

    article_summary = f"""
Title: {title}
Category: {category}
Section: {section}
Published: {pub_date}
URL: {url}
"""
    retrieved_texts.append(article_summary.strip())

In [ ]:
!pip install langchain langchain_openai langgraph

In [ ]:
from langchain_core.tools import Tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

In [ ]:
model = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
all_articles = "\n\n".join(retrieved_texts)

tools = [
    Tool(
        name="news_articles_context",
        func=lambda x: all_articles,
        description="Provides combined context from all retrieved news articles."
    )
]

In [ ]:
agent = create_react_agent(model, tools=tools)

In [ ]:
query = "What are the opinions on tariffs?"

In [ ]:
agent_response = agent.invoke({"messages": [("human", query)]})

In [ ]:
pprint.pp(agent_response['messages'][-1].content)